## 1. Imports e load dei dati

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = Path("../data/synthetic")

df = pd.read_csv(DATA_DIR / "synthetic_assessments.csv")
df_gt = pd.read_csv(DATA_DIR / "synthetic_assessments_ground_truth.csv")

print(df.shape)
print(df_gt.shape)

df.head()

## 2. Check base: colonne, tipi, missing

In [ ]:
print("Observed columns:")
print(df.columns.tolist())

print("\nGround truth columns:")
print(df_gt.columns.tolist())

print("\nMissing values in observed data:")
display(df.isna().sum().sort_values(ascending=False))

print("\nMissing values in ground truth:")
display(df_gt.isna().sum().sort_values(ascending=False))

## 3. Join con ground truth

In [ ]:
df_full = df.merge(df_gt, on="assessment_id", how="left")

print(df_full.shape)
df_full.head()

## 4. Distribuzioni base delle variabili operative

In [ ]:
for col in ["admin1", "admin2", "population_group", "partner", "week"]:
    print(f"\n=== {col} ===")
    display(df_full[col].value_counts(dropna=False).to_frame("count"))

## 5. Distribuzione della severity class

In [ ]:
severity_counts = df_full["gt_severity_class"].value_counts().reindex(["Minimal", "Stress", "Severe", "Extreme", "Catastrophic"])
display(severity_counts.to_frame("count"))

severity_counts.plot(kind="bar", figsize=(8, 4))
plt.title("Ground Truth Severity Class Distribution")
plt.xlabel("Severity class")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

## 6. Frequenza delle need categories

In [ ]:
needs_exploded = (
    df_full[["assessment_id", "gt_need_categories"]]
    .assign(gt_need_categories=lambda x: x["gt_need_categories"].str.split("|"))
    .explode("gt_need_categories")
)

need_counts = needs_exploded["gt_need_categories"].value_counts()
display(need_counts.to_frame("count"))

need_counts.plot(kind="bar", figsize=(10, 4))
plt.title("Ground Truth Need Category Frequency")
plt.xlabel("Need category")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

## 7. Frequenza delle urgent categories

In [ ]:
urgent_exploded = (
    df_full[["assessment_id", "gt_urgent_categories"]]
    .assign(
        gt_urgent_categories=lambda x: x["gt_urgent_categories"]
        .fillna("")
        .apply(lambda s: s.split("|") if s else [])
    )
    .explode("gt_urgent_categories")
)

urgent_exploded = urgent_exploded[
    urgent_exploded["gt_urgent_categories"].notna() &
    (urgent_exploded["gt_urgent_categories"] != "")
]

urgent_counts = urgent_exploded["gt_urgent_categories"].value_counts()
display(urgent_counts.to_frame("count"))

urgent_counts.plot(kind="bar", figsize=(10, 4))
plt.title("Ground Truth Urgent Need Category Frequency")
plt.xlabel("Urgent need category")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

## 8. Check qualitativo dei testi

In [ ]:
sample_cols = [
    "assessment_id",
    "admin1",
    "admin2",
    "population_group",
    "needs",
    "urgent_needs",
    "notes",
    "gt_need_categories",
    "gt_urgent_categories",
    "gt_severity_class",
]

display(df_full[sample_cols].sample(10, random_state=42))
display(df_full.loc[df_full["gt_severity_class"] == "Extreme", sample_cols].sample(min(10, (df_full["gt_severity_class"] == "Extreme").sum()), random_state=42))
display(df_full.loc[df_full["gt_severity_class"] == "Stress", sample_cols].sample(10, random_state=42))

In [ ]:
display(
    df_full.loc[df_full["gt_severity_class"].isin(["Extreme", "Catastrophic"]), sample_cols]
    .sample(10, random_state=7)
)

## 9. Check lunghezza dei testi

In [ ]:
df_full["notes_len"] = df_full["notes"].fillna("").str.len()
df_full["needs_len"] = df_full["needs"].fillna("").str.len()
df_full["urgent_needs_len"] = df_full["urgent_needs"].fillna("").str.len()

df_full[["notes_len", "needs_len", "urgent_needs_len"]].describe()
df_full["notes_len"].hist(bins=20, figsize=(8, 4))
plt.title("Distribution of Notes Length")
plt.xlabel("Characters")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

## 10. Pattern geografici: severity per area geografica

In [ ]:
severity_by_admin2 = (
    df_full.groupby(["admin1", "admin2"])["gt_severity_score"]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

display(severity_by_admin2)
severity_by_admin2["label"] = severity_by_admin2["admin1"] + " / " + severity_by_admin2["admin2"]

severity_by_admin2.plot(
    kind="bar",
    x="label",
    y="gt_severity_score",
    figsize=(12, 4),
    legend=False
)
plt.title("Average Ground Truth Severity Score by Admin2")
plt.xlabel("Area")
plt.ylabel("Average severity score")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 11. Pattern temporali: severity nel tempo

In [ ]:
severity_by_week = df_full.groupby("week")["gt_severity_score"].mean().reset_index()
display(severity_by_week)

plt.figure(figsize=(8, 4))
plt.plot(severity_by_week["week"], severity_by_week["gt_severity_score"], marker="o")
plt.title("Average Ground Truth Severity Score by Week")
plt.xlabel("Week")
plt.ylabel("Average severity score")
plt.xticks(severity_by_week["week"])
plt.tight_layout()
plt.show()

## Quick validation checklist

- Are the text fields realistic enough?
- Are there enough severe and critical cases?
- Are need categories sufficiently diverse?
- Are there visible geographic patterns?
- Are there visible temporal patterns?
- Is the language too repetitive?
- Do we need more ambiguity/noise before testing the LLM?

## Check on baseline features

In [ ]:
from pathlib import Path
import pandas as pd

df_base = pd.read_parquet(Path("../data/processed/baseline_features.parquet"))

df_base[[
    "assessment_id",
    "pred_need_categories",
    "gt_need_categories",
    "pred_severity_class",
    "gt_severity_class"
]].head(20)

In [ ]:
df_base["pred_severity_class"].value_counts()

In [ ]:
pd.crosstab(
    df_base["gt_severity_class"],
    df_base["pred_severity_class"],
    margins=True
)

In [ ]:
import numpy as np
y_true = df_base["gt_severity_class"]
y_pred = df_base["pred_severity_class"]

labels = sorted(set(y_true) | set(y_pred))
cm = pd.crosstab(y_true, y_pred).reindex(index=labels, columns=labels, fill_value=0)

# Normalizzazione per riga (ogni classe vera somma a 1)
cm_norm = cm.div(cm.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)

fig, ax = plt.subplots(figsize=(6, 6))
im = ax.imshow(cm_norm.values, cmap="Blues", vmin=0, vmax=1)

ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha="right")
ax.set_yticklabels(labels)
ax.set_xlabel("Predicted")
ax.set_ylabel("Ground Truth")
ax.set_title("Confusion Matrix (Normalized by True Class)")

# Mostra percentuali nelle celle
for i in range(cm_norm.shape[0]):
    for j in range(cm_norm.shape[1]):
        ax.text(j, i, f"{cm_norm.iloc[i, j]:.1%}", ha="center", va="center", color="black")

cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Proportion")

plt.tight_layout()
plt.savefig("../outputs/figures/confusion_matrix_baseline_normalized.png", dpi=150)
plt.show()